In [5]:
#This script is kept for documentaion, this has been implemented into src/0_data_cleaning
import pandas as pd

In [ ]:
path = '../data/raw/U.S._Chronic_Disease_Indicators__CDI___2023_Release copy.csv'
df_all = pd.read_csv(path)

In [ ]:
# Drop Columns from entire dataset that are missing = Num of missing values == num of rows --> Completely empty
df = df_all.drop(columns=[
            'Response',  # Entire col is empty
            'StratificationCategory2', # Entire col is empty
            'Stratification2', # Entire col is empty
            'StratificationCategory3', # Entire col is empty
            'Stratification3', # Entire col is empty
            'ResponseID', # Entire col is empty
            'StratificationCategoryID2', # Entire col is empty
            'StratificationID2', # Entire col is empty
            'StratificationCategoryID3', # Entire col is empty
            'StratificationID3', # Entire col is empty
            'YearEnd', # We will use Year start instead
            'LocationAbbr', # We will use location instead
            'DataSource', #Not needed at this moment 
            'DataValueAlt', #Not needed, will use DataValue 
            'DataValueFootnoteSymbol', #Not needed at this moment 
            'DatavalueFootnote', #Not needed at this moment 
            'HighConfidenceLimit', #Not needed at this moment 
            'LowConfidenceLimit', #Not needed at this moment 
            'LocationID', # We will use Location
            'DataValueTypeID',  # Not needed 
            'TopicID' # we know this is copd
        ])
print(df.shape)

In [8]:
# Now filter entier dataset for topic = 'Chronic Obstructive Pulmonary Disease'
topic = 'Chronic Obstructive Pulmonary Disease'
df = df[df['Topic'] == topic]

In [9]:
states_to_keep = ["Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", "Delaware", 
                  "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky", 
                  "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota", "Mississippi", "Missouri", 
                  "Montana", "Nebraska", "Nevada", "New Hampshire", "New Jersey", "New Mexico", "New York", "North Carolina", 
                  "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota", 
                  "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"]

df = df[df['LocationDesc'].isin(states_to_keep)]

In [10]:
# Drop All Unecessary Columns
df = df.drop(columns=[
            'Topic',
            'GeoLocation',
            'QuestionID',
            'StratificationCategoryID1',
            'StratificationID1',
            'DataValueUnit',
            'StratificationCategory1'
])

In [11]:
df = df[df['DataValueType'] == 'Number']

In [12]:
df = df[df['Question'] == 'Mortality with chronic obstructive pulmonary disease as underlying or contributing cause among adults aged >= 45 years']

In [13]:
df = df.dropna()

In [14]:
df = df.drop(columns=['DataValueType']) #drop data value type since we know this is NUMBER OF MORTALITY NOW

In [15]:
# Rename Columns to 
df.rename(columns={'YearStart': 'Year',
                    'LocationDesc': 'State',
                    'DataValue': 'Mortality',
                    'Stratification1': 'Stratification'}, inplace=True)

In [16]:
df = df[df['Year'] != 2010]

In [17]:
#Sort by state and year
df.sort_values(['State', 'Year'], inplace=True)

In [18]:
output_dir = '/home/daniel.lien/dev/homework/data/final_check/cleaned_copd_data_final.csv'
df.to_csv(output_dir, index=False)

In [19]:
medicaid_path = '/home/daniel.lien/dev/homework/Chronic_Disease_Indicators_Data_Visualization/workspaces/ashley/MEDICAID_AGGREGATE20.CSV'

In [ ]:
df = pd.read_csv(medicaid_path)
df.head()

In [21]:
cols_to_drop = [
    'Code',
    'Region_Number',
    'Region_Name',
    'Y1991', 'Y1992', 'Y1993', 'Y1994','Y1995','Y1996','Y1997','Y1998','Y1999', 'Y2000', 'Y2001','Y2002','Y2003','Y2004','Y2005','Y2006','Y2007','Y2008','Y2009','Y2010',
    'Average_Annual_Percent_Growth'
]
df = df.drop(columns=cols_to_drop)

In [22]:
# Filter by 'Region' == 'State Name'
df = df[df['Group'] == 'State']

In [23]:
df = df.dropna()

In [24]:
df = df[df['State_Name'] != 'District of Columbia']

In [25]:
year_columns = [col for col in df.columns if col.startswith("Y")]  # Identifying year columns
df[year_columns] = df[year_columns].apply(pd.to_numeric, errors='coerce')

In [26]:
# Melt the DataFrame to long format
df_long = df.melt(id_vars=['State_Name'], value_vars=year_columns, 
                   var_name='Year', value_name='Expenses')

# Convert 'Year' to integer
df_long['Year'] = df_long['Year'].str.extract('(\\d+)').astype(int)

# Group by state and year, summing Medicaid expenses
df_grouped = df_long.groupby(['State_Name', 'Year'])['Expenses'].sum().reset_index()

In [27]:
# Rename Columns to 
df_grouped.rename(columns={'YearStart': 'Year',
                    'State_Name': 'State',
                    'Expenses': 'Medicaid_Spending',
                    }, inplace=True)

In [28]:
output_dir = '/home/daniel.lien/dev/homework/data/final_check/medicaid_data_cleaned.csv'
df_grouped.to_csv(output_dir, index=False)

In [29]:
median_income = '/home/daniel.lien/dev/homework/data/h08.xlsx'

In [30]:
df = pd.read_excel(median_income, skiprows=7)

In [31]:
#pandas automatically handled multi index 
df.columns = df.columns.map(str)
df = df.loc[:, ~df.columns.str.contains('^Unnamed: ')]

In [32]:
years_to_drop = ['2017 (40)', '2013 (39)']
df = df.drop(columns=years_to_drop)

In [33]:
import re
def extract_year(col):
    match = re.search(r'\b(19|20)\d{2}\b', str(col))
    return match.group(0) if match else col

df.columns = [extract_year(col) for col in df.columns]

In [ ]:
#Drop
years_to_keep = [str(year) for year in range(2011,2021)]
#my 2013 is weird and 2020 is weird 
columns_to_keep = ['State'] + years_to_keep if 'State' in df.columns else years_to_keep
df = df[columns_to_keep]
df.head()

In [ ]:
df = df.drop(index=0)
df.head()

In [ ]:
# Drop rows where 'State' is 'USA' or 'District of Columbia'
df = df[~df['State'].isin(['United States', 'District of Columbia'])]

df.head()

In [ ]:
# Only keep the first half, take first 51 rows
df = df.iloc[:50]
df.shape

In [ ]:
df.columns

In [39]:
df_long = df.melt(id_vars='State', var_name='Year', value_name='Median_Income')

In [ ]:

df_long.head()

In [ ]:
df_long.columns

In [42]:
output_dir = '/home/daniel.lien/dev/homework/data/final_check/medianincome_temp.csv'
df_long.to_csv(output_dir, index=False)

In [43]:
medicaid = '/home/daniel.lien/dev/homework/data/final_check/medicaid_data_cleaned.csv'
medicaid_df = pd.read_csv(medicaid)
median_income = '/home/daniel.lien/dev/homework/data/final_check/medianincome_temp.csv'
median_income_df = pd.read_csv(median_income)  
copd = '/home/daniel.lien/dev/homework/data/final_check/cleaned_copd_data_final.csv'
copd_df = pd.read_csv(copd)

In [44]:
# Merge median income into COPD data
df_merged = copd_df.merge(median_income_df, on=['State', 'Year'], how='left')

# Merge Medicare spending into the result
df_merged = df_merged.merge(medicaid_df, on=['State', 'Year'], how='left')

In [45]:
df_merged = df_merged.drop(columns='Question')

In [ ]:
df_merged.columns

In [47]:
# Re-Order
df_merged = df_merged[['Year', 'State', 'Stratification', 'Medicaid_Spending', 'Median_Income', 'Mortality']]

In [48]:
output_dir = '/home/daniel.lien/dev/homework/data/final_check/final_csv.csv'
df_merged.to_csv(output_dir, index=False)